# ForenSURE-Net: Reliability-Calibrated Steganalysis

This notebook runs the full training, calibration, and triage pipeline for ForenSURE-Net on the BOSSbase 1.01 dataset.

## 1. Setup Environment
Install required dependencies and create directories.

In [ ]:
!pip install -r requirements.txt
import os

os.makedirs("data/BOSSBase/cover", exist_ok=True)
os.makedirs("data/BOSSBase/stego", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("experiments/checkpoints", exist_ok=True)
os.makedirs("experiments/calibration", exist_ok=True)

## 2. Link Kaggle Dataset
Symlink the BOSSbase dataset into the expected local path `data/BOSSBase/cover`. Note: Depending on your Kaggle environment, you may need to adjust the source path to match where your dataset is mounted.

In [ ]:
import glob
import shutil
from pathlib import Path

kaggle_path = "/kaggle/input/datasets/narendrachahar/bossbase-1-01/BOSSbase_1.01"
local_cover = "data/BOSSBase/cover"

if os.path.exists(kaggle_path):
    print("Copying dataset from Kaggle path...")
    for file in glob.glob(f"{kaggle_path}/*.*"):
        shutil.copy(file, local_cover)
    print(f"Copied {len(os.listdir(local_cover))} cover images.")
else:
    print("Kaggle dataset path not found. Ensure dataset is attached to notebook.")

## 3. Generate Stego Images (If necessary)
If your dataset does not include stego images, run the generation script to create them.

In [ ]:
if len(os.listdir("data/BOSSBase/stego")) == 0 and len(os.listdir("data/BOSSBase/cover")) > 0:
    print("Generating Stego images using LSB...")
    !python scripts/generate_lsb_stego.py
else:
    print("Stego images already exist.")

## 4. Run Full Pipeline
This executes the data splitting, model training, evaluation, uncertainty quantification, and report generation.

In [ ]:
!python scripts/run_full_pipeline.py

## 5. Review Results
Check the generated metrics and triage report.

In [ ]:
import json
import pandas as pd

try:
    with open("results/residual_stegnet_test_results.json", "r") as f:
        results = json.load(f)
    print("\nTest Results Summary:")
    for k, v in results.items():
        if k not in ["confusion_matrix", "reliability_bins"]:
            print(f"{k}: {v}")
            
    with open("results/uncertainty_triage_outputs.json", "r") as f:
        triage = json.load(f)
    print("\nTop 5 Suspicious Triage Items:")
    print(pd.DataFrame(triage[:5]))
except FileNotFoundError:
    print("Results files not found. Ensure the pipeline ran successfully.")